In [ ]:
from transformers import AutoTokenizer, AutoProcessor, SiglipTextModel, SiglipVisionModel

siglip_text_model = SiglipTextModel.from_pretrained("google/siglip-base-patch16-256")
#siglip_text_model.to(device)
total_params = sum(p.numel() for p in siglip_text_model.parameters())
print(f"Total parameters: {total_params/1e6:.2f}M")
siglip_text_model.to('cuda:1')
siglip_vision_model = SiglipVisionModel.from_pretrained("google/siglip-base-patch16-256")
#siglip_vision_model.to(device)
total_params = sum(p.numel() for p in siglip_vision_model.parameters())
print(f"Total parameters: {total_params/1e6:.2f}M")
siglip_vision_model.to('cuda:1')
siglip_tokenizer = AutoTokenizer.from_pretrained("google/siglip-base-patch16-256")
siglip_processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-256")

In [ ]:
!pip install transformers
!pip install git+https://github.com/openai/CLIP.git
!pip install --upgrade google-api-core>=2.10.2,<3.0.0dev
!pip install clipcap
!pip install open_clip_torch==2.32.0
!pip install fairscale
!pip install -r requirements.txt -q
!pip uninstall -y timm
!pip install timm==0.4.12
!pip install tensorboardX
!pip install torchscale
!pip install sacremoses

In [ ]:
import open_clip
import torch
from transformers import (GPT2Tokenizer, GPT2LMHeadModel)
import torch
import clip
import requests
from PIL import Image
import os
from torch import nn
import numpy as np
import torch.nn.functional as nnf
from typing import Tuple, List, Union, Optional, Callable
from transformers import GPT2Tokenizer, GPT2LMHeadModel, get_linear_schedule_with_warmup
from tqdm import tqdm, trange
import skimage.io as io
import PIL.Image
from IPython.display import Image 
from copy import deepcopy
import torchvision.transforms as transforms 
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib as mpl
import torchvision.datasets as datasets
from torchvision.transforms.functional import InterpolationMode
import imageio
from skimage.transform import resize
from transformers import pipeline, set_seed
from kaggle_secrets import UserSecretsClient
import torchvision.transforms as transforms
import glob
import torch
import cv2
from PIL import Image
from tqdm import tqdm
import sys
from transformers import AlignProcessor, AlignModel
import torchvision.transforms.functional as TF
import torchvision.transforms as T
from types import MethodType
import torch.nn.functional as F
from transformers.image_utils import ChannelDimension

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
align_processor = AlignProcessor.from_pretrained("kakaobrain/align-base")
align_model = AlignModel.from_pretrained("kakaobrain/align-base")
#align_model.to(device)
total_params = sum(p.numel() for p in align_model.parameters())
print(f"Total parameters: {total_params/1e6:.2f}M")
align_model.to('cuda:1')

In [ ]:
!git clone https://github.com/fonzi22/unilm.git /kaggle/working/BEIT3

import sys
sys.path.append('/kaggle/working/BEIT3/beit3')

In [ ]:
import modeling_finetune
from torchvision import transforms
from torchvision.datasets.folder import default_loader
from transformers import XLMRobertaTokenizer
from timm import create_model
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD, IMAGENET_INCEPTION_MEAN, IMAGENET_INCEPTION_STD
from IPython.display import clear_output

In [ ]:
checkpoint = torch.load('/kaggle/input/beit3-weights/beit3_large_patch16_384_coco_retrieval.pth')
beit3_model = create_model('beit3_large_patch16_384_retrieval')
beit3_model.load_state_dict(checkpoint['model'])
#beit3_model.eval().to(device)
total_params = sum(p.numel() for p in beit3_model.parameters())
print(f"Total parameters: {total_params/1e6:.2f}M")
beit3_model.eval().to('cuda:0')
beit3_tokenizer = XLMRobertaTokenizer('/kaggle/input/beit3-spm/beit3.spm')
#clear_output()

In [ ]:
!pip install salesforce-lavis
#!pip install --upgrade transformers
from lavis.models import load_model_and_preprocess

In [ ]:
!git clone https://github.com/salesforce/LAVIS
%cd LAVIS
!pip install .
!pip3 install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.0.0/en_core_web_sm-3.0.0.tar.gz
%cd projects/img2prompt-vqa

In [ ]:
img2prompt_model, img2prompt_vis_processors, img2prompt_txt_processors = load_model_and_preprocess(name="img2prompt_vqa", model_type="base", is_eval=True, device='cpu')
del img2prompt_txt_processors

In [ ]:
from transformers import GenerationConfig
def deterministic_caption(image, max_length=20):
    tokenizer = img2prompt_model.image_captioning_model.tokenizer

    encoder_out = img2prompt_model.image_captioning_model.visual_encoder(image)
    image_atts = torch.ones(encoder_out.size()[:-1], dtype=torch.long).to(image.device)

    # Greedy generation config
    gen_config = GenerationConfig(
        max_length=max_length, min_length=0, do_sample=False,
        num_beams=1, no_repeat_ngram_size=0, repetition_penalty=1.0, pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id, bos_token_id=tokenizer.bos_token_id)

    # Generate using BLIP's text decoder (pass vision features as encoder_hidden_states)
    caption_ids = img2prompt_model.image_captioning_model.text_decoder.generate(
        encoder_hidden_states=encoder_out, encoder_attention_mask=image_atts, generation_config=gen_config,)

    caption = tokenizer.decode(caption_ids[0], skip_special_tokens=True)
    return caption

In [ ]:
albef_model, albef_vis_processors, albef_txt_processors = load_model_and_preprocess(name="albef_feature_extractor", model_type="base", is_eval=False, device='cuda:1')
total_params = sum(p.numel() for p in albef_model.parameters())
print(f"Total parameters: {total_params/1e6:.2f}M")
albef_model.to('cuda:1')

In [ ]:
import torch.nn.functional as F
from lavis.common.registry import registry
from lavis.common.utils import get_abs_path
from lavis.models.albef_models import AlbefBase
from lavis.models.albef_models.albef_outputs import AlbefOutputFeatures
from lavis.models.med import BertForMaskedLM
from lavis.models.vit import VisionTransformerEncoder
from torch import nn
from transformers import BertConfig
import warnings

def custom_extract_features(self, samples, mode="multimodal"):
    # Copy the original extract_features code without @torch.no_grad()
    image = samples["image"]
    caption = samples["text_input"]
    if isinstance(mode, str):
        mode = [mode]
    for m in mode:
        assert m in ["multimodal", "image", "text"], f"Mode must be one of [multimodal, image, text], got {m}"
    image_embeds, text_embeds, multimodal_embeds = None, None, None
    image_features, text_features = None, None
    if "image" in mode or "multimodal" in mode:
        assert image is not None, "Image must be provided for 'image' or 'multimodal' mode"
        image_embeds = self.visual_encoder.forward_features(image)
        image_features = F.normalize(self.vision_proj(image_embeds), dim=-1)
    if "text" in mode or "multimodal" in mode:
        assert caption is not None, "Text must be provided for 'text' or 'multimodal' mode"
        text = self.tokenizer(caption, padding=True, return_tensors="pt").to(self.device)
        text_output = self.text_encoder.bert(
            text.input_ids, attention_mask=text.attention_mask, return_dict=True, mode="text"
        )
        text_embeds = text_output.last_hidden_state
        text_features = F.normalize(self.text_proj(text_embeds), dim=-1)
    if "multimodal" in mode:
        image_atts = torch.ones(image_embeds.size()[:-1], dtype=torch.long).to(self.device)
        output = self.text_encoder.bert(
            encoder_embeds=text_embeds,
            attention_mask=text.attention_mask,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_atts,
            return_dict=True,
            mode="fusion",
        )
        multimodal_embeds = output.last_hidden_state
    return AlbefOutputFeatures(
        image_embeds=image_embeds,
        image_embeds_proj=image_features,
        text_embeds=text_embeds,
        text_embeds_proj=text_features,
        multimodal_embeds=multimodal_embeds,
    )

albef_model.extract_features = MethodType(custom_extract_features, albef_model)

In [ ]:
import sys
!git clone https://github.com/salesforce/BLIP.git /kaggle/working/BLIP

sys.path.append('/kaggle/working/BLIP')

In [ ]:
class MLP(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)
    def __init__(self, sizes: Tuple[int, ...], bias=True, act=nn.Tanh):
        super(MLP, self).__init__()
        layers = []
        for i in range(len(sizes)-1):
            layers.append(nn.Linear(sizes[i], sizes[i+1], bias=bias))
            if i < len(sizes) - 2:
                layers.append(act())
        self.model = nn.Sequential(*layers)

class ClipCaptionModel(nn.Module):
    def get_dummy_token(self, batch_size: int, device: torch.device) -> torch.Tensor:
        return torch.zeros(batch_size, self.prefix_length, dtype=torch.int64, device=device)
        
    def forward(self, tokens: torch.Tensor, prefix: torch.Tensor, mask: Optional[torch.Tensor] = None, labels: Optional[torch.Tensor] = None):
        embedding_text = self.gpt.transformer.wte(tokens)
        prefix_projections = self.clip_project(prefix).view(-1, self.prefix_length, self.gpt_embedding_size)
        embedding_cat = torch.cat((prefix_projections, embedding_text), dim=1)
        if labels is not None:
            dummy_token = self.get_dummy_token(tokens.shape[0], tokens.device)
            labels = torch.cat((dummy_token, tokens), dim=1)
        out = self.gpt(inputs_embeds=embedding_cat, labels=labels, attention_mask=mask)
        return out
        
    def __init__(self, prefix_length: int, prefix_size: int = 512):
        super(ClipCaptionModel, self).__init__()
        self.prefix_length = prefix_length
        self.gpt = GPT2LMHeadModel.from_pretrained('gpt2')
        self.gpt_embedding_size = self.gpt.transformer.wte.weight.shape[1]
        if prefix_length > 10: 
            self.clip_project = nn.Linear(prefix_size, self.gpt_embedding_size * prefix_length)
        else:
            self.clip_project = MLP((prefix_size, (self.gpt_embedding_size*prefix_length) //2 , self.gpt_embedding_size * prefix_length))

class ClipCaptionPrefix(ClipCaptionModel):
    def parameters(self, recurse: bool = True):
        return self.clip_project.parameters()

    def train(self, mode: bool=True):
        super(ClipCaptionPrefix, self).train(mode)
        self.gpt.eval()
        return self
        
def generate_beam(model, tokenizer, beam_size: int=5, prompt=None, embed=None, entry_length=67, temperature=1., stop_token: str = '.'):
    model.eval()
    stop_token_index = tokenizer.encode(stop_token)[0]
    tokens = None
    scores = None
    device = next(model.parameters()).device
    # seq_lengths = torch.ones(beam_size, device=device)
    seq_lengths = torch.ones(beam_size, device='cuda:1')
    # is_stopped = torch.zeros(beam_size, device=device, dtype=torch.bool)
    is_stopped = torch.zeros(beam_size, device='cuda:1', dtype=torch.bool)
    with torch.no_grad():
        if embed is None:
            generated = embed
        else:
            if token is None:
                tokens = torch.tensor(tokenizer.encode(prompt))
                # tokens = tokens.unsqueeze(0).to(device)
                tokens = tokens.unsqueeze(0).to('cuda:1')
                generated = model.gpt.transformer.wte(tokens)

        for i in range(entry_length):
            outputs = model.gpt(inputs_embeds=generated)
            logits = outputs.logits
            logits = logits[:, -1, :] /  (temperature if temperature > 0 else 1.0)
            logits = logits.softmax(-1).log()
            if scores is None:
                scores, next_tokens = logits.topk(beam_size, -1)
                generated = generated.expand(beam_size, *generated.shape[1:])
                next_tokens, scores = next_tokens.permute(1, 0), scores.squeeze(0)
                if tokens is None:
                    tokens = next_tokens
                else:
                    tokens = tokens.expand(beam_size, *tokens.shape[1:])
                    tokens = torch.cat((tokens, next_tokens), dim=1)

            else:
                logits[is_stopped] = -float(np.inf)
                logits[is_stopped, 0] = 0
                scores_sum = scores[:, None] + logits
                seq_lengths[~is_stopped] += 1
                scores_sum_average = scores_sum / seq_lengths[:, None]
                scores_sum_average, next_tokens = scores_sum_average.view(-1).topk(beam_size, -1)
                next_tokens_source = next_tokens // scores_sum.shape[1]
                seq_lengths = seq_lengths[next_tokens_source]
                next_tokens = next_tokens % scores_sum.shape[1]
                next_tokens = next_tokens.unsqueeze(1)
                tokens = tokens[next_tokens_source]
                tokens = torch.cat((tokens, next_tokens), dim = 1)
                generated = generated[next_tokens_source]
                scores = scores_sum_average*seq_lengths
                is_stopped = is_stopped[next_tokens_source]

            next_token_embed = model.gpt.transformer.wte(next_tokens.squeeze()).view(generated.shape[0], 1, -1)
            generated = torch.cat((generated, next_token_embed), dim = 1)
            is_stopped = is_stopped + next_tokens_eq(stop_token_index).squeeze()
            if is_stopped.all():
                break
    scores = scores/seq_lengths
    output_list = tokens.cpu().numpy()
    output_texts = [tokenizer.decode(output[:int(length)]) for output, length in zip(output_list, seq_lengths)]
    order = scores.argsort(descending=True)
    output_texts = [output_texts[i] for i in order]
    return output_texts
def generate2(model, tokenizer, tokens=None, prompt=None, embed=None, entry_count=1, entry_length=67, top_p=0.8, temperature=1., stop_token: str = '.'):
    model.eval()
    generated_num = 0
    generated_list = []
    stop_token_index = tokenizer.encode(stop_token)[0]
    filter_value = -float("Inf")
    device = next(model.parameters()).device

    with torch.no_grad():
        for entry_idx in range(entry_count):
            if embed is not None:
                generated = embed
            else:
                if tokens is None:
                    tokens = torch.tensor(tokenizer.encode(prompt))
                    # tokens = tokens.unsqueeze(0).to(device)
                    tokens = tokens.unsqueeze(0).to('cuda:1')

                generated = model.gpt.transformer.wte(tokens)

            for i in range(entry_length):
                outputs = model.gpt(inputs_embeds=generated)
                logits = outputs.logits
                logits = logits[:, -1, :] / (temperature if temperature > 0 else 1.0)
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(nnf.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0

                indices_to_remove = sorted_indices[sorted_indices_to_remove]
                logits[:, indices_to_remove] = filter_value
                next_token = torch.argmax(logits, -1).unsqueeze(0)
                next_token_embed = model.gpt.transformer.wte(next_token)
                if tokens is None:
                    tokens = next_token
                else:
                    tokens = torch.cat((tokens, next_token), dim=1)
                generated = torch.cat((generated, next_token_embed), dim=1)
                if stop_token_index == next_token.item():
                    break
            output_list = list(tokens.squeeze().cpu().numpy())
            output_text = tokenizer.decode(output_list)
            generated_list.append(output_text)
            
    return generated_list[0]

In [ ]:
model_path = '/kaggle/input/clip-model/model_wieghts.pt'

In [ ]:
# clip_vit_B32_model, preprocess = clip.load("ViT-B/32", device=device, jit=False)
clip_vit_B32_model, preprocess = clip.load("ViT-B/32", device='cuda:1', jit=False)
total_params = sum(p.numel() for p in clip_vit_B32_model.parameters())
print(f"Total parameters: {total_params/1e6:.2f}M")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

prefix_length = 10

caption_model = ClipCaptionModel(prefix_length)
state_dict = torch.load(model_path, map_location=torch.device('cpu'), weights_only=True)
caption_model.load_state_dict(state_dict, strict=False)

total_params = sum(p.numel() for p in caption_model.parameters())
print(f"Total parameters: {total_params/1e6:.2f}M")

caption_model = caption_model.eval()
#caption_model = caption_model.to(device)
caption_model = caption_model.to('cuda:1')

In [ ]:
from models.blip import blip_decoder
from models.blip import blip_feature_extractor

captioning_model_path = '/kaggle/input/another-checkpoint/model_base_capfilt_large (1).pth'
med_config_path = '/kaggle/working/BLIP/configs/med_config.json'

blip_model_captioning = blip_decoder(pretrained=captioning_model_path, med_config = med_config_path, image_size=384, vit='base')
blip_model_captioning.eval()
#blip_model_captioning.to(device)
total_params = sum(p.numel() for p in blip_model_captioning.parameters())
print(f"Total parameters: {total_params/1e6:.2f}M")

feature_extractor_model_path = '/kaggle/input/feature-extracting-model/model_base.pth'
blip_model_feature_extractor = blip_feature_extractor(pretrained=feature_extractor_model_path, med_config = med_config_path, image_size=384, vit='base')
blip_model_feature_extractor.eval()
#blip_model_feature_extractor.to(device)
total_params = sum(p.numel() for p in blip_model_feature_extractor.parameters())
print(f"Total parameters: {total_params/1e6:.2f}M")
blip_model_feature_extractor.to('cuda:0')

In [ ]:
def blip(image, blip_model_captioning=blip_model_captioning, image_size=384):
    image = image.unsqueeze(0)
    blip_model_captioning.eval()
    #blip_model_captioning = blip_model_captioning.to(device)
    blip_model_captioning = blip_model_captioning.to('cuda:0')

    with torch.no_grad():
        # beam search
        caption = blip_model_captioning.generate(image, sample=False, num_beams=1, max_length=50, min_length=5)
        #caption = model.generate(image, sample=True, num_beams=5, max_length=50, min_length=5)
    return caption[0]

In [ ]:
#clip_rn_101_model, preprocess_rn_101 = clip.load("RN101", device=device, jit=False)
clip_rn_101_model, preprocess_rn_101 = clip.load("RN101", device='cuda:1', jit=False)
clip_rn_101_model.eval()
#clip_rn_101_model.to(device)
total_params = sum(p.numel() for p in clip_rn_101_model.parameters())
print(f"Total parameters: {total_params/1e6:.2f}M")
clip_rn_101_model.to('cuda:1')

#clip_vit_B16_model, preprocess_vit_B16 = clip.load("ViT-B/16", device=device, jit=False)
clip_vit_B16_model, preprocess_vit_B16 = clip.load("ViT-B/16", device='cuda:1', jit=False)
clip_vit_B16_model.eval()
#clip_vit_B16_model.to(device)
total_params = sum(p.numel() for p in clip_vit_B16_model.parameters())
print(f"Total parameters: {total_params/1e6:.2f}M")
clip_vit_B16_model.to('cuda:1')

#clip_vit_L14_model, preprocess_vit_L14 = clip.load("ViT-L/14", device=device, jit=False)
clip_vit_L14_model, preprocess_vit_L14 = clip.load("ViT-L/14", device='cuda:0', jit=False)
clip_vit_L14_model.eval()
#clip_vit_L14_model.to(device)
total_params = sum(p.numel() for p in clip_vit_L14_model.parameters())
print(f"Total parameters: {total_params/1e6:.2f}M")
clip_vit_L14_model.to('cuda:0')

In [ ]:
coca_model, _, coca_transform = open_clip.create_model_and_transforms(model_name="coca_ViT-L-14", pretrained="mscoco_finetuned_laion2B-s13B-b90k")
#coca_model = coca_model.to(device)
total_params = sum(p.numel() for p in coca_model.parameters())
print(f"Total parameters: {total_params/1e6:.2f}M")
coca_model = coca_model.to('cuda:1')
coca_tokenizer = open_clip.get_tokenizer("coca_ViT-L-14")

In [ ]:
import shutil
from transformers import utils as transformers_utils

In [ ]:
transformers_modules = [key for key in sys.modules if 'transformers' in key]
for module_name in transformers_modules:
    sys.modules.pop(module_name, None)

cache_dir = os.getenv("TRANSFORMERS_CACHE", os.path.expanduser("~/.cache/huggingface"))
if os.path.exists(cache_dir):
    try:
        shutil.rmtree(cache_dir)
        print(f"Cleared transformers cache at {cache_dir}")
    except Exception as e:
        print(f"Error clearing cache: {e}")
else:
    print(f"No cache directory found at {cache_dir}")

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
torch.cuda.manual_seed_all(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
albef_model.eval() 
albef_model.zero_grad(set_to_none=True) 

In [ ]:
def tensor_siglip_processor(image_tensor, size=(256, 256), resample='bicubic', do_rescale=True, rescale_factor=1/255, do_normalize=True, 
                    image_mean=[0.5, 0.5, 0.5], image_std=[0.5, 0.5, 0.5], data_format=ChannelDimension.FIRST):
    
    if image_tensor.shape[0] == 3:
        image_tensor = image_tensor.unsqueeze(0)
    if image_tensor.ndim != 4:
        raise ValueError(f"Expected 4D tensor (B,C,H,W), got {image_tensor.shape}")
        
    B, C, H, W = image_tensor.shape
    if C != 3:
        raise ValueError(f"Expected 3 channels, got {C}")
        
    if image_tensor.dtype != torch.float32:
        image_tensor = image_tensor.to(torch.float32)

    min_val, max_val = image_tensor.min().item(), image_tensor.max().item()
    if max_val <= 1.1 and min_val >= -0.1:
        image_tensor = image_tensor * 255
    elif min_val < 0 or max_val > 255.1:
        print(f"Warning: Input range [{min_val}, {max_val}] not in [0, 255]")

    image_tensor = image_tensor.requires_grad_(True)

    resized_image = F.interpolate(image_tensor, size=size, mode=resample, align_corners=False, antialias=True)

    processed_image = resized_image
    if do_rescale:
        processed_image = processed_image * rescale_factor

    if do_normalize:
        mean = torch.tensor(image_mean, device=processed_image.device).view(1, C, 1, 1)
        std = torch.tensor(image_std, device=processed_image.device).view(1, C, 1, 1)
        processed_image = (processed_image - mean) / std

    if data_format == ChannelDimension.LAST:
        processed_image = processed_image.permute(0, 2, 3, 1)

    return processed_image

def tensor_align_processor(image_tensor, size=(346, 346), crop_size=(289, 289), do_center_crop=True, do_rescale=True, rescale_factor=0.00784313725490196, rescale_offset=True, do_normalize=False, image_mean=[0.5, 0.5, 0.5], image_std=[0.5, 0.5, 0.5], data_format=ChannelDimension.FIRST):

    if image_tensor.ndim == 3:
        image_tensor = image_tensor.unsqueeze(0)
    if image_tensor.ndim != 4:
        raise ValueError(f"Expected 4D tensor (B,C,H,W), got {image_tensor.shape}")

    b, c, h, w = image_tensor.shape
    if c != 3:
        raise ValueError(f"Expected 3 channels, got {c}")
    if image_tensor.dtype != torch.float32:
        print(f"Warning: Converting dtype from {image_tensor.dtype} to float32")
        image_tensor = image_tensor.to(torch.float32)

    min_val, max_val = image_tensor.min().item(), image_tensor.max().item()
    if max_val <= 1.1 and min_val >= -0.1:  # Likely [0, 1]
        #print("Input range appears to be [0, 1], scaling to [0, 255]")
        image_tensor = image_tensor * 255
    elif min_val < 0 or max_val > 255.1:
        print(f"Warning: Input range [{min_val}, {max_val}] not in [0, 255]")

    image_tensor = image_tensor.requires_grad_(True)

    resized_image = F.interpolate(image_tensor, size=size, mode='bilinear', align_corners=False)
    #print(f"After resize to {size}: min={resized_image.min().item()}, max={resized_image.max().item()}")

    if do_center_crop:
        _, _, h, w = resized_image.shape
        crop_height, crop_width = crop_size
        top = (h - crop_height) // 2
        left = (w - crop_width) // 2

        if h < crop_height or w < crop_width:
            pad_top = max(0, (crop_height - h) // 2)
            pad_bottom = max(0, crop_height - h - pad_top)
            pad_left = max(0, (crop_width - w) // 2)
            pad_right = max(0, crop_width - w - pad_left)
            resized_image = F.pad(resized_image, (pad_left, pad_right, pad_top, pad_bottom), mode='constant', value=0)
            _, _, h, w = resized_image.shape
            top = (h - crop_height) // 2
            left = (w - crop_width) // 2

        processed_image = resized_image[:, :, top:top + crop_height, left:left + crop_width]
        #print(f"After crop to {crop_size}: min={processed_image.min().item()}, max={processed_image.max().item()}")
    else:
        processed_image = resized_image

    # 3. Rescale
    if do_rescale:
        processed_image = processed_image * rescale_factor
        #print(f"After rescale (*{rescale_factor}): min={processed_image.min().item()}, max={processed_image.max().item()}")
        if rescale_offset:
            processed_image = processed_image - 1
            #print(f"After offset (-1): min={processed_image.min().item()}, max={processed_image.max().item()}")

    if do_normalize:
        mean = torch.tensor(image_mean, device=processed_image.device).view(1, c, 1, 1)
        std = torch.tensor(image_std, device=processed_image.device).view(1, c, 1, 1)
        processed_image = (processed_image - mean) / std

    if data_format == ChannelDimension.LAST:
        processed_image = processed_image.permute(0, 2, 3, 1)

    return processed_image

def tensor_coca_transform(image_size: Union[int, Tuple[int, int]] = 224, mean: Tuple[float, float, float] = (0.48145466, 0.4578275, 0.40821073), std: Tuple[float, float, float] = (0.26862954, 0.26130258, 0.27577711), interpolation: str = 'bicubic') -> callable:

    if isinstance(image_size, int):
        image_size = (image_size, image_size)
        
    target_h, target_w = image_size

    interpolation_modes = {
        'bilinear': TF.InterpolationMode.BILINEAR,
        'bicubic': TF.InterpolationMode.BICUBIC,
        'nearest': TF.InterpolationMode.NEAREST,}
    interpolation_mode = interpolation_modes.get(interpolation, TF.InterpolationMode.BICUBIC)

    def transform(tensor: torch.Tensor) -> torch.Tensor:
        if tensor.ndim == 3:
            tensor = tensor.unsqueeze(0)
        if tensor.ndim != 4:
            raise ValueError(f"Expected 4D tensor (B,C,H,W), got shape {tensor.shape}")
        if tensor.shape[1] not in (1, 3, 4):
            raise ValueError(f"Expected 1, 3, or 4 channels, got {tensor.shape[1]} channels")

        if tensor.dtype == torch.uint8:
            tensor = tensor.float() / 255.0
        elif tensor.dtype in (torch.float32, torch.float16):
            if tensor.max() > 1.0 + 1e-5 or tensor.min() < 0.0 - 1e-5:
                raise ValueError("Float tensor values must be in [0, 1]")
        else:
            raise ValueError(f"Unsupported tensor dtype: {tensor.dtype}")

        # h, w = tensor.shape[1], tensor.shape[2]
        b, c, h, w = tensor.shape
        scale = max(target_h / h, target_w / w)  # Shortest-side resizing
        new_h, new_w = int(h * scale), int(w * scale)
        tensor = TF.resize(tensor, size=[new_h, new_w], interpolation=interpolation_mode)

        # Center Crop to target size
        tensor = TF.center_crop(tensor, output_size=[target_h, target_w])

        # Normalize
        tensor = TF.normalize(tensor, mean=mean, std=std)

        return tensor
        
    return transform
    

def tensor_albef_vis_processor(image_size: Union[int, Tuple[int, int]] = 224, mean: Tuple[float, float, float] = (0.48145466, 0.4578275, 0.40821073),
                                    std: Tuple[float, float, float] = (0.26862954, 0.26130258, 0.27577711), interpolation: str = 'bicubic') -> Callable:

    if isinstance(image_size, int):
        image_size = (image_size, image_size)
    target_h, target_w = image_size

    # Map interpolation string to torchvision enum
    interpolation_modes = {
            'bilinear': TF.InterpolationMode.BILINEAR,
            'bicubic': TF.InterpolationMode.BICUBIC,
            'nearest': TF.InterpolationMode.NEAREST,
        }
    interpolation_mode = interpolation_modes.get(interpolation, TF.InterpolationMode.BICUBIC)

    def transform(tensor: torch.Tensor) -> torch.Tensor:
        if tensor.ndim == 3:
            tensor = tensor.unsqueeze(0)
        if tensor.ndim != 4:
            raise ValueError(f"Expected 4D tensor (B,C,H,W), got shape {tensor.shape}")
        if tensor.shape[1] not in (1, 3, 4):
            raise ValueError(f"Expected 1, 3, or 4 channels, got {tensor.shape[1]} channels")

        if tensor.dtype == torch.uint8:
            tensor = tensor.float() / 255.0
        elif tensor.dtype in (torch.float32, torch.float16):
            if tensor.max() > 1.0 + 1e-5 or tensor.min() < 0.0 - 1e-5:
                raise ValueError("Float tensor values must be in [0, 1]")
        else:
            raise ValueError(f"Unsupported tensor dtype: {tensor.dtype}")

        #Resize to target size
        tensor = TF.resize(tensor, size=[target_h, target_w], interpolation=interpolation_mode)

        # Normalize
        tensor = TF.normalize(tensor, mean=mean, std=std)

        return tensor

    return transform

In [ ]:
def get_embedding_clip_vit_b32(image: torch.Tensor=None, text: str = None) -> torch.Tensor:
    if image is not None:
        embedding = clip_vit_B32_model.encode_image(image.to('cuda:1')).to('cuda:1', dtype=torch.float32)
        embedding = embedding / embedding.norm(dim=1, keepdim=True)
        #embedding = caption_model.clip_project(embedding).reshape(1, prefix_length, -1)
        #embedding = embedding.mean(dim=1)  
        return embedding
    if text is not None:
        tokens = clip.tokenize([text]).to('cuda:1')
        embedding = clip_vit_B32_model.encode_text(tokens).to('cuda:1', dtype=torch.float32)
        embedding = embedding / embedding.norm(dim=1, keepdim=True)

        #embedding = caption_model.clip_project(embedding).reshape(1, prefix_length, -1)
        #embedding = embedding.mean(dim=1)  
        return embedding
    
def get_embedding_clip_rn_101(image: torch.Tensor=None, text: str = None) -> torch.Tensor:
    if image is not None:
        embedding = clip_rn_101_model.encode_image(image.to('cuda:1')).to('cuda:1', dtype=torch.float32)
        embedding = embedding / embedding.norm(dim=1, keepdim=True)

        return embedding
            
    if text is not None:
        tokens = clip.tokenize([text]).to('cuda:1')
        embedding = clip_rn_101_model.encode_text(tokens).to('cuda:1', dtype=torch.float32)
        embedding = embedding / embedding.norm(dim=1, keepdim=True)

        return embedding

def get_embedding_clip_vit_b16(image: torch.Tensor=None, text: str = None) -> torch.Tensor:
    if image is not None:
        embedding = clip_vit_B16_model.encode_image(image.to('cuda:1')).to('cuda:1', dtype=torch.float32)
        embedding = embedding / embedding.norm(dim=1, keepdim=True)
        return embedding
            
    if text is not None:
        tokens = clip.tokenize([text]).to('cuda:1')
        embedding = clip_vit_B16_model.encode_text(tokens).to('cuda:1', dtype=torch.float32)
        embedding = embedding / embedding.norm(dim=1, keepdim=True)
        return embedding

def get_embedding_clip_vit_l14(image: torch.Tensor=None, text: str = None) -> torch.Tensor:
    if image is not None:
        embedding = clip_vit_L14_model.encode_image(image.to('cuda:0')).to('cuda:0', dtype=torch.float32)
        embedding = embedding / embedding.norm(dim=1, keepdim=True)

        #del image
        torch.cuda.empty_cache()
            
        return embedding
            
    if text is not None:
        tokens = clip.tokenize([text]).to('cuda:0')
        embedding = clip_vit_L14_model.encode_text(tokens).to('cuda:0', dtype=torch.float32)
        embedding = embedding / embedding.norm(dim=1, keepdim=True)
        return embedding
    
def get_embedding_albef(image: torch.Tensor=None, text: str = None) -> torch.Tensor:
    if image is not None:
        #to_pil = transforms.ToPILImage()
        #pil_image = to_pil(image.squeeze())
            
        albef_tensor_processors = tensor_albef_vis_processor(
            image_size=224,
            mean=(0.48145466, 0.4578275, 0.40821073),
            std=(0.26862954, 0.26130258, 0.27577711),
            interpolation='bicubic')

        image_for_albef = albef_tensor_processors(image).to('cuda:1')
        #image_for_albef = albef_vis_processors["eval"](pil_image).unsqueeze(0).to('cuda:1') 
        image_for_albef = {"image": image_for_albef, "text_input": None}
        embedding = albef_model.extract_features(image_for_albef, mode="image")
        embedding = embedding.image_embeds_proj[:, 0, :]
        embedding = embedding / embedding.norm(dim=1, keepdim=True)
        return embedding
        
    if text is not None:
        text_for_albef = albef_txt_processors["eval"](text)
        text_for_albef = {"image": None, "text_input": [text_for_albef]}
        embedding = albef_model.extract_features(text_for_albef, mode="text")
        embedding = embedding.text_embeds_proj[:, 0, :]
        embedding = embedding / embedding.norm(dim=1, keepdim=True)
        return embedding
    
def get_embedding_blip(image: torch.Tensor=None, text: str = None) -> torch.Tensor:
    # kỳ vọng image là [1, 3, 224, 224]
    if image is not None:

        resize_384 = transforms.Resize((384, 384), interpolation=transforms.InterpolationMode.BILINEAR)
        image_clone = image.clone()
        image_clone = resize_384(image_clone)

        out = blip_model_feature_extractor(image_clone.to('cuda:0'), "", mode='image')
        embedding = out[:, 0, :]
        embedding = embedding / embedding.norm(dim=1, keepdim=True)

        # del resize_384, image_clone
        torch.cuda.empty_cache()

        return embedding
            
    if text is not None:
        embedding = blip_model_feature_extractor(torch.empty(1, 3, 384, 384).to('cuda:0'), ori_text, mode='text')[0,0]
        embedding = embedding.unsqueeze(0)
        embedding = embedding / embedding.norm(dim=1, keepdim=True)

        return embedding
    
def get_embedding_coca(image: torch.Tensor=None, text: str = None) -> torch.Tensor:
    if image is not None:
        #to_pil = transforms.ToPILImage()
        #pil_image = to_pil(image.squeeze())
        #image_for_coca = coca_transform(pil_image).unsqueeze(0).to('cuda:1')

        coca_transform_for_tensor = tensor_coca_transform(
            image_size=224,
            mean=(0.48145466, 0.4578275, 0.40821073),
            std=(0.26862954, 0.26130258, 0.27577711),
            interpolation='bicubic')
            
        image_for_coca = coca_transform_for_tensor(image).to('cuda:1')    
        embedding = coca_model.encode_image(image_for_coca)
        #del image_for_coca  
        #torch.cuda.empty_cache()
        embedding = embedding / embedding.norm(dim=1, keepdim=True)
        return embedding

    if text is not None:
        text_for_coca = coca_tokenizer([text]).to('cuda:1')
        embedding = coca_model.encode_text(text_for_coca)
        embedding = embedding / embedding.norm(dim=1, keepdim=True)
        return embedding
    
def get_embedding_beit3(image: torch.Tensor=None, text: str = None) -> torch.Tensor:
    if image is not None:
        resize_384 = transforms.Resize((384, 384), interpolation=transforms.InterpolationMode.BILINEAR)
        image_clone = image.clone()
        image_clone = resize_384(image_clone)

        embedding = beit3_model(image=image_clone.to('cuda:0'), only_infer=True)[0]
        embedding = embedding / embedding.norm(dim=1, keepdim=True)

        # del resize_384, image_clone
        torch.cuda.empty_cache()
            
        return embedding

    if text is not None:
        def beit3_get_text_segment(text, tokenizer, max_len = 64):
            tokens = tokenizer.tokenize(text)
            tokens = tokenizer.convert_tokens_to_ids(tokens)
            if len(tokens) > max_len -2:
                tokens = tokens[:max_len-2]
            tokens = [tokenizer.bos_token_id] + tokens[:] + [tokenizer.eos_token_id]
            num_tokens = len(tokens)
            padding_mask = [0] * num_tokens + [1] * (max_len - num_tokens)
            language_tokens = tokens + [tokenizer.pad_token_id] * (max_len - num_tokens)
            return torch.tensor([language_tokens]),  torch.tensor([padding_mask]), num_tokens

        def encode_text_query_beit3(query, model, tokenizer):
            language_tokens, padding_mask, _ = beit3_get_text_segment(query, tokenizer)
            language_tokens = language_tokens.to(device)
            padding_mask = padding_mask.to(device)
            _, language_cls = model(text_description=language_tokens, padding_mask=padding_mask, only_infer=True)

            result = language_cls
            return result
            
        embedding = encode_text_query_beit3(text, beit3_model, beit3_tokenizer)
        embedding = embedding / embedding.norm(dim=1, keepdim=True)

        return embedding
    
def get_embedding_align(image: torch.Tensor=None, text: str=None) -> torch.Tensor:
    if image is not None:
        #to_pil = transforms.ToPILImage()
        #pil_image = to_pil(image.squeeze())

        #image_input_align = align_processor(images=pil_image, return_tensors="pt").to('cuda:1')
        #embedding = align_model.get_image_features(pixel_values=image_input_align['pixel_values'])
        image_input_align = tensor_align_processor(image.to('cuda:1'), size=(align_processor.image_processor.size['height'], align_processor.image_processor.size['width']),
            crop_size=(align_processor.image_processor.crop_size['height'], align_processor.image_processor.crop_size['width']), do_center_crop=align_processor.image_processor.do_center_crop, do_rescale=align_processor.image_processor.do_rescale,
            rescale_factor=align_processor.image_processor.rescale_factor, rescale_offset=align_processor.image_processor.rescale_offset, do_normalize=align_processor.image_processor.do_normalize, image_mean=align_processor.image_processor.image_mean,
            image_std=align_processor.image_processor.image_std, data_format=ChannelDimension.FIRST)
            
        embedding = align_model.get_image_features(pixel_values=image_input_align)

        #del image_input_align
        #torch.cuda.empty_cache()
            
        embedding = embedding / embedding.norm(dim=1, keepdim=True)
        return embedding

    if text is not None:
        text_input_align = align_processor(text=text, return_tensors="pt").to('cuda:1')
        embedding = align_model.get_text_features(input_ids=text_input_align['input_ids'], 
                                                           attention_mask=text_input_align['attention_mask'], 
                                                           token_type_ids=text_input_align['token_type_ids'])

        #del text_input_align
        #torch.cuda.empty_cache()  
            
        embedding = embedding / embedding.norm(dim=1, keepdim=True)
        return embedding
    
def get_embedding_siglip(image: torch.Tensor=None, text: str=None) -> torch.Tensor:
    if image is not None:
        #to_pil = transforms.ToPILImage()
        #pil_image = to_pil(image.squeeze())
        #image_input_siglip = siglip_processor(images=pil_image, return_tensors="pt")
        #image_input_siglip = image_input_siglip.to('cuda:1')

        image_input_siglip = tensor_siglip_processor(image.to('cuda:1'), size=(256, 256),
            resample='bicubic', do_rescale=True, rescale_factor=1/255, do_normalize=True, image_mean=[0.5, 0.5, 0.5], image_std=[0.5, 0.5, 0.5], data_format=ChannelDimension.FIRST)
        #image_output_siglip = siglip_vision_model(**image_input_siglip)
        image_output_siglip = siglip_vision_model(pixel_values=image_input_siglip)

        #del image_input_siglip
        #torch.cuda.empty_cache()  
            
        embedding = image_output_siglip.pooler_output
        embedding = embedding / embedding.norm(dim=1, keepdim=True)
        return embedding
            
    if text is not None:
        text_input_siglip = siglip_tokenizer([text], padding="max_length", return_tensors="pt")
        text_input_siglip = text_input_siglip.to('cuda:1')
        text_output_siglip = siglip_text_model(**text_input_siglip)
        embedding = text_output_siglip.pooler_output
        embedding = embedding / embedding.norm(dim=1, keepdim=True)
        return embedding

In [ ]:

import os, random
from PIL import Image
from tqdm import tqdm

import numpy as np
import torch
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import joblib

# --- Cấu hình ---
image_path = '/kaggle/input/coco-2017-dataset/coco2017/test2017'
random.seed(42)
ALL_JPG = [os.path.join(image_path, f) for f in os.listdir(image_path) if f.lower().endswith('.jpg')]
jpg_files = random.sample(ALL_JPG, 40000)
train_files = jpg_files[:35000]
test_files = jpg_files[35000:]

batch_size = 128          # thử 32/64/128 tùy GPU
num_workers = 8         # thử 4/8; Kaggle CPU core giới hạn, tinh chỉnh
pin_memory = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_fp16 = True         # bật mixed precision nếu GPU hỗ trợ

# --- Dataset & transforms ---
transform = T.Compose([
    T.Resize((224,224)),   # nếu model cần size khác thì handle riêng
    T.ToTensor(),
    # (optional) normalize theo mean/std model nếu bạn muốn chính xác
])

class ImagePathDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        path = self.paths[idx]
        with Image.open(path) as img:
            img = img.convert('RGB')
            tensor = self.transform(img)
        return tensor  # shape (C,H,W)

train_ds = ImagePathDataset(train_files, transform)
test_ds  = ImagePathDataset(test_files, transform)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=pin_memory)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=pin_memory)

models = [
    ("clip_vit_b32", get_embedding_clip_vit_b32),
    ("clip_rn_101", get_embedding_clip_rn_101),
    ("clip_vit_b16", get_embedding_clip_vit_b16),
    ("clip_vit_l14", get_embedding_clip_vit_l14),
    ("blip", get_embedding_blip),
    ("coca", get_embedding_coca),
    ("albef", get_embedding_albef),
    ("align", get_embedding_align),
    ("siglip", get_embedding_siglip),
    ("beit3", get_embedding_beit3),
]

# --- Utility: detect embedding dims by running one small batch once ---
print("Detecting embedding dims with a sample batch...")
sample_batch = next(iter(train_loader)).to(device)
dims = {}
with torch.no_grad():
    with torch.cuda.amp.autocast(enabled=use_fp16):
        for name, fn in models:
            out = fn(image=sample_batch)   # ensure fn supports batch, else wrap
            out = out.detach().cpu().numpy()
            out = out.reshape(out.shape[0], -1)  # (B, d)
            dims[name] = out.shape[1]
print("Detected dims:", dims)

# --- Create memmap files to store embeddings incrementally ---
os.makedirs("embeddings_memmap", exist_ok=True)
N_train = len(train_ds)
N_test = len(test_ds)
memmaps_train = {}
memmaps_test = {}
for name, d in dims.items():
    memmaps_train[name] = np.memmap(f"embeddings_memmap/{name}_train.dat", dtype='float32', mode='w+', shape=(N_train, d))
    memmaps_test[name]  = np.memmap(f"embeddings_memmap/{name}_test.dat",  dtype='float32', mode='w+', shape=(N_test, d))

# --- Function to compute embeddings for a batch and write to memmap ---
def compute_and_write(loader, memmaps, total_N):
    idx0 = 0
    for batch in tqdm(loader, desc="Batches"):
        B = batch.size(0)
        batch = batch.to(device, non_blocking=True)
        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=use_fp16):
                for name, fn in models:
                    # call fn on the whole batch; ensure it returns (B, d)
                    emb = fn(image=batch)  # expected torch tensor (B, d) or (B, d, 1,1)
                    emb = emb.detach().cpu().numpy().reshape(B, -1).astype('float32')
                    memmaps[name][idx0:idx0+B] = emb 

        idx0 += B
    # flush to disk
    for m in memmaps.values():
        m.flush()

# --- Run for train and test ---
print("Processing train set...")
compute_and_write(train_loader, memmaps_train, N_train)
print("Processing test set...")
compute_and_write(test_loader, memmaps_test, N_test)

print("All embeddings are saved to embeddings_memmap/*.dat")

In [ ]:
# Deep MCCA training: CCA objective + variance regularization (PyTorch)
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
import torch, torch.nn as nn, torch.optim as optim
import joblib, os, numpy as np

save_dir = "/kaggle/working/embeddings_memmap/DeepMCCA"
os.makedirs(save_dir, exist_ok=True)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --- Prepare scaled embeddings per view (same as you had) ---
scalers = {}
Xs_train, Xs_test = [], []

for name in dims.keys():
    X_train = np.array(memmaps_train[name])
    X_test  = np.array(memmaps_test[name])

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    scalers[name] = scaler
    Xs_train.append(X_train_scaled.astype(np.float32))
    Xs_test.append(X_test_scaled.astype(np.float32))

model_names = list(dims.keys())
print("Shapes per view:")
for n, X in zip(model_names, Xs_train):
    print(f"{n}: {X.shape}")

for name, X in zip(model_names, Xs_train):
    print(f"{name}: mean norm = {np.mean(np.linalg.norm(X, axis=1)):.3f}, var per dim = {X.var(axis=0).mean():.3e}")

# Dataset & loader (each sample is a tuple of view embeddings)
Xs_train_tensors = [torch.from_numpy(X) for X in Xs_train]
train_dataset = TensorDataset(*Xs_train_tensors)
batch_size = 1024
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

# --- Encoder per view (MLP) ---
class ViewEncoder(nn.Module):
    def __init__(self, in_dim, latent_dim=256, hidden=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, latent_dim),
        )
    def forward(self, x):
        return self.net(x)

latent_dim = 256
encoders = {
    name: ViewEncoder(in_dim=X.shape[1], latent_dim=latent_dim).to(device)
    for name, X in zip(model_names, Xs_train)
}

# optimizer
params = []
for enc in encoders.values():
    params += list(enc.parameters())
optimizer = optim.Adam(params, lr=1e-4, weight_decay=1e-5)

# -------------------------
# Helper linear algebra ops
# -------------------------
def symmetric_inverse_sqrt(mat, eps=1e-6):
    # mat: (d,d), symmetric PSD
    # returns mat^{-1/2} robustly
    vals, vecs = torch.linalg.eigh(mat)  # vals ascending
    vals = torch.clamp(vals, min=eps)
    inv_sqrt = (vecs * (1.0 / torch.sqrt(vals))).matmul(vecs.T)
    return inv_sqrt

def canonical_corrs_between(A, B, eps=1e-6, top_k=None):
    """
    A, B: (N, d) centered (torch tensors)
    returns singular values (canonical correlations) as 1D tensor (sorted desc)
    """
    N = A.shape[0]
    # covariance matrices
    C_aa = (A.T @ A) / (N - 1)
    C_bb = (B.T @ B) / (N - 1)
    C_ab = (A.T @ B) / (N - 1)

    # regularize and compute whitening inv-sqrt
    C_aa += eps * torch.eye(C_aa.shape[0], device=C_aa.device)
    C_bb += eps * torch.eye(C_bb.shape[0], device=C_bb.device)

    W_a = symmetric_inverse_sqrt(C_aa, eps=eps)
    W_b = symmetric_inverse_sqrt(C_bb, eps=eps)

    T = W_a @ C_ab @ W_b  # (d,d)
    # compute singular values of T
    # use torch.linalg.svdvals
    s = torch.linalg.svdvals(T)  # descending
    if top_k is not None:
        s = s[:top_k]
    return s

# -------------------------
# Deep CCA loss + variance reg
# -------------------------
def deep_cca_loss(latents, eps=1e-6, var_reg_coeff=1e-3, top_k=64):
    """
    latents: list of tensors (batch, latent_dim)
    returns scalar loss (to minimize)
    - cca_term: negative mean of sum of top-k canonical correlations for each pair
    - var_term: Frobenius norm penalty pushing covariance -> I (keeps variance)
    """
    n_views = len(latents)
    batch_size = latents[0].shape[0]
    d = latents[0].shape[1]
    # center
    latents = [z - z.mean(dim=0, keepdim=True) for z in latents]

    cca_terms = []
    cov_mats = []
    # compute pairwise canonical correlations
    for i in range(n_views):
        for j in range(i+1, n_views):
            s = canonical_corrs_between(latents[i], latents[j], eps=eps, top_k=top_k)  # tensor
            # sum or mean of top-k canonical correlations
            cca_terms.append(torch.mean(s))  

    if len(cca_terms) == 0:
        cca_loss = torch.tensor(0.0, device=latents[0].device)
    else:
        cca_loss = - torch.mean(torch.stack(cca_terms))  # maximize correlation => minimize negative

    # variance regularization: encourage cov ~ identity (preserve variance & decorrelate dims)
    var_losses = []
    for z in latents:
        C = (z.T @ z) / (batch_size - 1)
        I = torch.eye(d, device=C.device)
        var_losses.append(torch.norm(C - I, p='fro') / (d**0.5))  # normalized
    var_loss = torch.mean(torch.stack(var_losses))

    loss = cca_loss + var_reg_coeff * var_loss
    return loss, cca_loss.detach(), var_loss.detach()

# -------------------------
# Training loop
# -------------------------
epochs = 40   # tăng epochs (bắt đầu với 30)
print_every = 1

for epoch in range(1, epochs+1):
    tot_loss = 0.0
    tot_cca = 0.0
    tot_var = 0.0
    n_batches = 0
    for batch in train_loader:
        # batch is tuple of tensors per view
        batch = [x.to(device) for x in batch]
        latents = [encoders[name](x) for name, x in zip(model_names, batch)]

        loss, cca_term, var_term = deep_cca_loss(latents, eps=1e-6, var_reg_coeff=1e-3, top_k=64)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        tot_loss += loss.item()
        tot_cca += cca_term.item()
        tot_var += var_term.item()
        n_batches += 1

    if epoch % print_every == 0:
        print(f"Epoch {epoch}/{epochs} | loss {tot_loss/n_batches:.4f} | cca_term {tot_cca/n_batches:.4f} | var_term {tot_var/n_batches:.4f}")

# --- Save artifact (encoders + scalers + meta) ---
artifact = {
    "encoders": {name: enc.cpu().state_dict() for name, enc in encoders.items()},
    "scalers": scalers,
    "model_names": model_names,
    "latent_dim": latent_dim,
    "encoder_config": {"hidden": 512},
}
artifact_path = os.path.join(save_dir, f"deepmcca_{latent_dim}_with_scalers_and_encoders.pkl")
joblib.dump(artifact, artifact_path)
print("Saved artifact:", artifact_path)

# -------------------------
# Evaluate: compute canonical correlations (top-k mean) on test set
# -------------------------
for enc in encoders.values():
    enc.to(device)
Xs_test_tensors = [torch.from_numpy(X).to(device) for X in Xs_test]

with torch.no_grad():
    Zs_test = [encoders[name](X) for name, X in zip(model_names, Xs_test_tensors)]
    Zs_test = [z.cpu().numpy() for z in Zs_test]

def mean_topk_cca(A_np, B_np, top_k=20):
    A = torch.from_numpy(A_np).float()
    B = torch.from_numpy(B_np).float()
    s = canonical_corrs_between(A, B, eps=1e-6, top_k=top_k)
    return float(torch.mean(s).cpu().item())

print("\nCross-view mean(top-64 canonical correlations) (test set):")
for i in range(len(model_names)):
    for j in range(i+1, len(model_names)):
        val = mean_topk_cca(Zs_test[i], Zs_test[j], top_k=64)
        print(f"{model_names[i]} ↔ {model_names[j]}: {val:.4f}")